# 03 Embeddings and Vector Store

This notebook covers generating dense embeddings for text chunks using `sentence-transformers` and building an index with `faiss` for efficient vector similarity search.

In [3]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 76.9 MB/s eta 0:00:00


In [4]:
# ============================================================
# 03 - EMBEDDINGS & VECTOR STORE
# Hybrid RAG Project
# ============================================================

import json
import pickle
from pathlib import Path

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully")


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_DIR = Path("/content/hybrid-rag-project")
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"
VECTOR_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)
print("Vector store:", VECTOR_DIR)

Libraries imported successfully
Project: /content/hybrid-rag-project
Processed: /content/hybrid-rag-project/data/processed
Vector store: /content/hybrid-rag-project/data/vector_store


In [5]:
# ============================================================
# LOAD PROCESSED DATA
# ============================================================

with open(
    PROCESSED_DIR / "sentence_records.json",
    "r",
    encoding="utf-8"
) as f:
    sentence_records = json.load(f)


with open(
    PROCESSED_DIR / "sentence_windows.json",
    "r",
    encoding="utf-8"
) as f:
    sentence_windows = json.load(f)


with open(
    PROCESSED_DIR / "parent_child_records.json",
    "r",
    encoding="utf-8"
) as f:
    parent_child_records = json.load(f)


with open(
    PROCESSED_DIR / "pmay_u.json",
    "r",
    encoding="utf-8"
) as f:
    pmay_data = json.load(f)


print("Data loaded successfully")

print("\nSentence records:", len(sentence_records))
print("Sentence windows:", len(sentence_windows))
print("Parent-child records:", len(parent_child_records))

FileNotFoundError: [Errno 2] No such file or directory: '/content/hybrid-rag-project/data/processed/sentence_records.json'

In [6]:
%cd /content/hybrid-rag-project
!git pull origin main

/content/hybrid-rag-project
fatal: not a git repository (or any of the parent directories): .git


In [7]:
%cd /content

!rm -rf hybrid-rag-project

!git clone https://github.com/Dhinesh-Manikandan/Hybrid-RAG-Project.git /content/hybrid-rag-project

/content
Cloning into '/content/hybrid-rag-project'...
remote: Enumerating objects: 52, done.
remote: Counting objects: 100% (52/52), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 52 (delta 14), reused 26 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (52/52), 16.70 KiB | 8.35 MiB/s, done.
Resolving deltas: 100% (14/14), done.


In [ ]:
# ============================================================
# 03 - EMBEDDINGS & VECTOR STORE
# Hybrid RAG Project
# ============================================================

# ============================================================
# 0. INSTALL REQUIRED PACKAGE
# ============================================================

!pip install -q faiss-cpu sentence-transformers


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import json
import pickle
from pathlib import Path

import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully")


# ============================================================
# 2. PROJECT PATHS
# ============================================================

PROJECT_DIR = Path("/content/hybrid-rag-project")

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

VECTOR_DIR = PROJECT_DIR / "data" / "vector_store"

VECTOR_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)
print("Vector Store:", VECTOR_DIR)


# ============================================================
# 3. VERIFY PROCESSED FILES
# ============================================================

required_files = [
    "pmay_u.json",
    "sentence_records.json",
    "sentence_windows.json",
    "parent_child_records.json"
]

print("\nChecking processed files...")

for filename in required_files:

    file_path = PROCESSED_DIR / filename

    if not file_path.exists():
        raise FileNotFoundError(
            f"Missing file: {file_path}"
        )

    print("✓", filename)

print("\nAll processed files are available.")


# ============================================================
# 4. LOAD PROCESSED DATA
# ============================================================

with open(
    PROCESSED_DIR / "pmay_u.json",
    "r",
    encoding="utf-8"
) as f:
    pmay_data = json.load(f)


with open(
    PROCESSED_DIR / "sentence_records.json",
    "r",
    encoding="utf-8"
) as f:
    sentence_records = json.load(f)


with open(
    PROCESSED_DIR / "sentence_windows.json",
    "r",
    encoding="utf-8"
) as f:
    sentence_windows = json.load(f)


with open(
    PROCESSED_DIR / "parent_child_records.json",
    "r",
    encoding="utf-8"
) as f:
    parent_child_records = json.load(f)


print("\nData loaded successfully")

print("Sentence records:", len(sentence_records))
print("Sentence windows:", len(sentence_windows))
print("Parent-child records:", len(parent_child_records))


# ============================================================
# 5. LOAD EMBEDDING MODEL
# ============================================================

print("\nLoading embedding model...")

embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2",
    device="cuda"
)

print("Embedding model loaded successfully")
print("Device:", embedding_model.device)


# ============================================================
# 6. TEST EMBEDDING MODEL
# ============================================================

test_text = sentence_records[3]["text"]

test_embedding = embedding_model.encode(
    test_text,
    convert_to_numpy=True
)

print("\nEmbedding test")
print("Text:")
print(test_text)

print("\nEmbedding shape:")
print(test_embedding.shape)

print("\nEmbedding dimension:", len(test_embedding))


# ============================================================
# 7. HELPER FUNCTION FOR FAISS INDEX
# ============================================================

def create_faiss_index(embeddings):
    """
    Creates a FAISS inner-product index.

    Embeddings are normalized before indexing,
    so inner product corresponds to cosine similarity.
    """

    embeddings = embeddings.astype("float32")

    dimension = embeddings.shape[1]

    index = faiss.IndexFlatIP(dimension)

    index.add(embeddings)

    return index


# ============================================================
# 8. NAIVE RAG
# ============================================================
#
# For the baseline Naive RAG, we use the complete sections
# as retrieval units.
#
# Each section becomes one document chunk.
#
# Example:
#
# DETAILS
# BENEFITS
# ELIGIBILITY
# APPLICATION PROCESS
# etc.
#
# This gives us a simple baseline for comparison.
# ============================================================

naive_records = []

for section_name, content in pmay_data.items():

    # Metadata fields are not retrieval chunks
    if section_name in [
        "scheme_name",
        "ministry",
        "categories",
        "source"
    ]:
        continue

    if not isinstance(content, str):
        continue

    if not content.strip():
        continue

    naive_records.append({
        "chunk_id": len(naive_records),
        "section": section_name,
        "text": content.strip()
    })


print("\nNaive RAG chunks:", len(naive_records))


# ============================================================
# 9. CREATE NAIVE RAG EMBEDDINGS
# ============================================================

naive_texts = [
    record["text"]
    for record in naive_records
]

print("\nCreating Naive RAG embeddings...")

naive_embeddings = embedding_model.encode(
    naive_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Naive embeddings created")
print("Shape:", naive_embeddings.shape)


# ============================================================
# 10. CREATE NAIVE RAG FAISS INDEX
# ============================================================

naive_index = create_faiss_index(
    naive_embeddings
)

print("\nNaive RAG FAISS index created")
print("Vectors:", naive_index.ntotal)
print("Dimension:", naive_embeddings.shape[1])


# ============================================================
# 11. SAVE NAIVE RAG INDEX + METADATA
# ============================================================

naive_index_path = (
    VECTOR_DIR / "naive_rag.index"
)

faiss.write_index(
    naive_index,
    str(naive_index_path)
)

with open(
    VECTOR_DIR / "naive_rag_metadata.pkl",
    "wb"
) as f:
    pickle.dump(
        naive_records,
        f
    )

print("\nNaive RAG index saved:")
print(naive_index_path)


# ============================================================
# 12. SENTENCE-WINDOW RETRIEVAL
# ============================================================
#
# We embed the WINDOW TEXT.
#
# Example:
#
# Previous sentence
# Target sentence
# Next sentence
#
# The target sentence remains available in metadata,
# while the complete window is used for retrieval.
# ============================================================

window_texts = [
    record["window_text"]
    for record in sentence_windows
]

print("\nCreating Sentence-Window embeddings...")

window_embeddings = embedding_model.encode(
    window_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Sentence-Window embeddings created")
print("Shape:", window_embeddings.shape)


# ============================================================
# 13. CREATE SENTENCE-WINDOW FAISS INDEX
# ============================================================

window_index = create_faiss_index(
    window_embeddings
)

print("\nSentence-Window FAISS index created")
print("Vectors:", window_index.ntotal)
print("Dimension:", window_embeddings.shape[1])


# ============================================================
# 14. SAVE SENTENCE-WINDOW INDEX + METADATA
# ============================================================

window_index_path = (
    VECTOR_DIR / "sentence_window.index"
)

faiss.write_index(
    window_index,
    str(window_index_path)
)

with open(
    VECTOR_DIR / "sentence_window_metadata.pkl",
    "wb"
) as f:
    pickle.dump(
        sentence_windows,
        f
    )

print("\nSentence-Window index saved:")
print(window_index_path)


# ============================================================
# 15. PARENT-CHILD RETRIEVAL
# ============================================================
#
# Important:
#
# We embed CHILD TEXT.
#
# When a child is retrieved, its corresponding
# PARENT TEXT can later be returned to the LLM.
#
# Retrieval:
#
# Query
#   ↓
# Child embedding search
#   ↓
# Relevant child
#   ↓
# Parent ID
#   ↓
# Parent context
# ============================================================

child_texts = [
    record["child_text"]
    for record in parent_child_records
]

print("\nCreating Parent-Child child embeddings...")

child_embeddings = embedding_model.encode(
    child_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Parent-Child embeddings created")
print("Shape:", child_embeddings.shape)


# ============================================================
# 16. CREATE PARENT-CHILD FAISS INDEX
# ============================================================

parent_child_index = create_faiss_index(
    child_embeddings
)

print("\nParent-Child FAISS index created")
print("Vectors:", parent_child_index.ntotal)
print("Dimension:", child_embeddings.shape[1])


# ============================================================
# 17. SAVE PARENT-CHILD INDEX + METADATA
# ============================================================

parent_child_index_path = (
    VECTOR_DIR / "parent_child.index"
)

faiss.write_index(
    parent_child_index,
    str(parent_child_index_path)
)

with open(
    VECTOR_DIR / "parent_child_metadata.pkl",
    "wb"
) as f:
    pickle.dump(
        parent_child_records,
        f
    )

print("\nParent-Child index saved:")
print(parent_child_index_path)


# ============================================================
# 18. SAVE EMBEDDING INFORMATION
# ============================================================

embedding_info = {
    "model": "all-MiniLM-L6-v2",
    "dimension": int(naive_embeddings.shape[1]),
    "device": str(embedding_model.device),
    "similarity": "cosine_similarity",
    "faiss_index_type": "IndexFlatIP"
}

with open(
    VECTOR_DIR / "embedding_info.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        embedding_info,
        f,
        indent=2
    )

print("\nEmbedding information saved.")


# ============================================================
# 19. TEST QUERY
# ============================================================
#
# Before moving to the RAG pipeline, test whether
# our vector indexes can retrieve relevant information.
# ============================================================

test_query = "Who is eligible for PMAY-U?"

query_embedding = embedding_model.encode(
    [test_query],
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")


# ============================================================
# 20. TEST NAIVE RAG RETRIEVAL
# ============================================================

k = 3

naive_scores, naive_indices = naive_index.search(
    query_embedding,
    k
)

print("\n" + "=" * 60)
print("NAIVE RAG RETRIEVAL")
print("=" * 60)

for rank, (score, index_id) in enumerate(
    zip(
        naive_scores[0],
        naive_indices[0]
    ),
    start=1
):

    record = naive_records[index_id]

    print(f"\nRank {rank}")
    print("Score:", float(score))
    print("Section:", record["section"])
    print("Text:", record["text"][:500])


# ============================================================
# 21. TEST SENTENCE-WINDOW RETRIEVAL
# ============================================================

window_scores, window_indices = window_index.search(
    query_embedding,
    k
)

print("\n" + "=" * 60)
print("SENTENCE-WINDOW RETRIEVAL")
print("=" * 60)

for rank, (score, index_id) in enumerate(
    zip(
        window_scores[0],
        window_indices[0]
    ),
    start=1
):

    record = sentence_windows[index_id]

    print(f"\nRank {rank}")
    print("Score:", float(score))
    print("Section:", record["section"])
    print("Target:", record["target_sentence"])
    print("Window:", record["window_text"])


# ============================================================
# 22. TEST PARENT-CHILD RETRIEVAL
# ============================================================

pc_scores, pc_indices = parent_child_index.search(
    query_embedding,
    k
)

print("\n" + "=" * 60)
print("PARENT-CHILD RETRIEVAL")
print("=" * 60)

for rank, (score, index_id) in enumerate(
    zip(
        pc_scores[0],
        pc_indices[0]
    ),
    start=1
):

    record = parent_child_records[index_id]

    print(f"\nRank {rank}")
    print("Score:", float(score))
    print("Section:", record["section"])
    print("Child:", record["child_text"])
    print("Parent:", record["parent_text"][:500])


# ============================================================
# 23. FINAL VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("EMBEDDING & VECTOR STORE COMPLETED")
print("=" * 60)

print("\nEmbedding model:")
print("all-MiniLM-L6-v2")

print("\nEmbedding dimension:")
print(naive_embeddings.shape[1])

print("\nIndexes:")
print("Naive RAG:", naive_index.ntotal, "vectors")
print("Sentence-Window:", window_index.ntotal, "vectors")
print("Parent-Child:", parent_child_index.ntotal, "vectors")

print("\nVector store files:")

for file in sorted(VECTOR_DIR.iterdir()):
    print(" -", file.name)

print("\nPipeline:")
print("""
Processed Documents
        ↓
Embedding Model
        ↓
 ┌──────┼──────────────┐
 ↓      ↓              ↓
Naive  Sentence-     Parent-
RAG    Window        Child
 ↓      ↓              ↓
FAISS  FAISS          FAISS
 └──────┼──────────────┘
        ↓
    Retrieval
""")